# 03 — Feature Engineering

**Input:** `clean_data/clean_data_{train,test}.csv` (output của [02_Cleaning.ipynb](./02_Cleaning.ipynb)).

**Output:** ma trận sparse `X_{train,test}.npz` + target log `y_{train,test}.npy` + transformer đã fit, lưu trong `features/` để [04_Modeling.ipynb](./04_Modeling.ipynb) dùng.

**Cấu trúc đặc trưng** (theo thứ tự cột trong X):

| # | Nhóm | Cột nguồn | Transform | Chiều |
|---|---|---|---|---|
| 1 | numeric | `years_exp`, `year` | SimpleImputer(median) + StandardScaler() | 2 |
| 2 | one_hot | `province`, `education_level`, `job_type`, `job_position` | OneHotEncoder(handle_unknown='ignore') | ~95 |
| 3 | industries | `industries_list` (split `\|`) | MultiLabelBinarizer | 51 |
| 4 | tfidf_job_title | `job_title` (text ngắn, signal cao) | TfidfVectorizer max=5k, **ngram=(1,2)**, min_df=5 | ~5,000 |
| 5 | tfidf_job_description | `job_description` | TfidfVectorizer max=10k, ngram=(1,1) | 10,000 |
| 6 | tfidf_requirements | `requirements` | TfidfVectorizer max=10k | ~9,000 |
| 7 | tfidf_benefits | `benefits` | TfidfVectorizer max=10k | ~8,000 |
| **Tổng** | | | | **~32,000** |

**Quy tắc:**
- Mọi transformer **fit trên train**, transform test (không leak).
- Target `y = log1p(expected_salary)` — giảm skew, đỡ outlier ở đuôi. Stage 4 sẽ inverse `expm1` khi báo cáo MAE/RMSE ở thang triệu.
- `year` giữ trong X; stage 4 sẽ slice cột theo `meta['groups']['numeric']` để so sánh có/không `year`.
- TF-IDF dùng `strip_accents=None` — **giữ dấu tiếng Việt** (dấu mang nghĩa).
- `job_title` được tách riêng vì là text ngắn, signal đậm đặc nhất (chức danh quyết định 30-50% lương).

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path.cwd().parent
CLEAN = ROOT / 'clean_data'
FEATURES = ROOT / 'features'
FEATURES.mkdir(exist_ok=True)

RANDOM_STATE = 42

# Text dài (job_description, requirements, benefits) — unigram, vocab lớn
TFIDF_PARAMS_LONG = dict(
    max_features=10_000,
    min_df=10,
    max_df=0.95,
    sublinear_tf=True,
    ngram_range=(1, 1),
    lowercase=True,
    strip_accents=None,  # giữ dấu tiếng Việt
)

# Text ngắn (job_title) — bigram, vocab nhỏ, min_df thấp hơn vì title ngắn
TFIDF_PARAMS_TITLE = dict(
    max_features=5_000,
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    ngram_range=(1, 2),   # bigram bắt cụm: "kế toán trưởng", "senior engineer"
    lowercase=True,
    strip_accents=None,
)

# (col, params) — thứ tự sẽ là thứ tự trong X cuối cùng
TFIDF_COLS_PARAMS = [
    ('job_title', TFIDF_PARAMS_TITLE),
    ('job_description', TFIDF_PARAMS_LONG),
    ('requirements', TFIDF_PARAMS_LONG),
    ('benefits', TFIDF_PARAMS_LONG),
]
TEXT_COLS = [col for col, _ in TFIDF_COLS_PARAMS]

CAT_COLS = ['province', 'education_level', 'job_type', 'job_position']
NUM_COLS = ['years_exp', 'year']

print(f"Output: {FEATURES}")

Output: D:\Documents\School\Ki_6\KHDL\09 - Dự đoán mức lương kỳ vọng dựa trên bản mô tả công việc (Job Description)\features


## 1. Đọc clean data

Pandas mặc định parse empty string thành NaN — fix bằng `fillna('')` cho cột text/multi-label, `fillna('unknown')` cho cột categorical (đề phòng trường hợp `02_Cleaning.ipynb` để sót).

In [2]:
def load_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in TEXT_COLS + ['industries_list']:
        df[col] = df[col].fillna('').astype(str)
    for col in CAT_COLS:
        df[col] = df[col].fillna('unknown').astype(str)
    return df


train = load_clean(CLEAN / 'clean_data_train.csv')
test = load_clean(CLEAN / 'clean_data_test.csv')

print(f"train: {train.shape} | test: {test.shape}")
print(f"\nNaN còn sót (train):")
miss = train.isna().sum()
print(miss[miss > 0].to_string() or '  (không cột nào NaN)')

train: (458568, 14) | test: (114773, 14)

NaN còn sót (train):
years_exp       9392
company_name      10


## 2. Target — `y = log1p(expected_salary)`

`log1p(x) = ln(1 + x)` kéo phân phối lương lệch phải về gần normal hơn → linear regression ổn định hơn, không phá đặc tính cộng tính của mô hình. Khi báo cáo MAE/RMSE ở thang triệu, dùng `np.expm1(y_pred)` để nghịch đảo.

In [3]:
y_train = np.log1p(train['expected_salary'].to_numpy(dtype=np.float64))
y_test = np.log1p(test['expected_salary'].to_numpy(dtype=np.float64))

print(f"y_train: shape={y_train.shape}, mean={y_train.mean():.3f}, std={y_train.std():.3f}")
print(f"y_test : shape={y_test.shape}, mean={y_test.mean():.3f}, std={y_test.std():.3f}")
print(f"\nLương gốc (triệu, train):")
print(train['expected_salary'].describe().round(2).to_string())

y_train: shape=(458568,), mean=2.574, std=0.413
y_test : shape=(114773,), mean=2.573, std=0.412

Lương gốc (triệu, train):
count    458568.00
mean         13.32
std           6.80
min           0.15
25%           9.00
50%          12.00
75%          15.00
max         100.00


### Trực quan — target trước và sau `log1p`

Phân phối lương gốc lệch phải mạnh (skew ≈ +2) → linear regression với MSE sẽ "nịnh" outlier ở đuôi cao. `log1p` kéo về phân phối gần normal, ổn định hơn cho Ridge/Lasso/LightGBM (vì MAE/MSE đều xét residual đối xứng).

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import skew

FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'

y_raw = train['expected_salary'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.hist(y_raw, bins=80, range=(0, 100), color='steelblue', alpha=0.85, edgecolor='white', linewidth=0.3)
ax.axvline(np.median(y_raw), color='red', linestyle='--', lw=1, label=f'median = {np.median(y_raw):.1f}tr')
ax.axvline(np.mean(y_raw), color='orange', linestyle='--', lw=1, label=f'mean = {np.mean(y_raw):.1f}tr')
ax.set_xlabel('expected_salary (triệu)')
ax.set_ylabel('Số JD')
ax.set_title(f'Trước — expected_salary  (skew = {skew(y_raw):+.2f})')
ax.legend()

ax = axes[1]
ax.hist(y_train, bins=80, color='seagreen', alpha=0.85, edgecolor='white', linewidth=0.3)
ax.axvline(np.median(y_train), color='red', linestyle='--', lw=1, label=f'median = {np.median(y_train):.2f}')
ax.axvline(np.mean(y_train), color='orange', linestyle='--', lw=1, label=f'mean = {np.mean(y_train):.2f}')
ax.set_xlabel('y = log1p(expected_salary)')
ax.set_ylabel('Số JD')
ax.set_title(f'Sau — log1p  (skew = {skew(y_train):+.2f})')
ax.legend()

plt.suptitle('Target transform — log1p giảm skew từ +2.x về gần 0', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fe_target_log_transform.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'fe_target_log_transform.png'}")

## 3. Numeric — `years_exp`, `year`

- `years_exp`: ~2% NaN (đã loại junk bucket `9.0` ở stage 2). Impute **median** fit trên train.
- `year`: không NaN, giữ nguyên (đã là int).
- `StandardScaler()` — **center và scale** cả hai cột. Lưu ý: nếu chỉ scale không center, `year` (giá trị 2022..2026) sau khi chia std ~1.17 sẽ có giá trị ~1730 → linear iterative solver (Lasso/SGD) blow up. Centering biến `year` về khoảng [-2, +2], an toàn cho mọi model.
- Khối numeric trở thành dense 2 cột × 459k = ~920k giá trị non-zero (chỉ +1% so với tổng 88M nnz của full X), không phá sparsity một cách có ý nghĩa.

In [4]:
num_imputer = SimpleImputer(strategy='median')
X_num_train = num_imputer.fit_transform(train[NUM_COLS])
X_num_test = num_imputer.transform(test[NUM_COLS])

print(f"Median (train): years_exp = {num_imputer.statistics_[0]}, year = {num_imputer.statistics_[1]}")

num_scaler = StandardScaler()  # center và scale — an toàn cho linear/SGD/Lasso
X_num_train = sparse.csr_matrix(num_scaler.fit_transform(X_num_train), dtype=np.float32)
X_num_test = sparse.csr_matrix(num_scaler.transform(X_num_test), dtype=np.float32)

print(f"X_num_train: {X_num_train.shape} | X_num_test: {X_num_test.shape}")
print(f"Numeric stats (train, sau scale):")
arr = X_num_train.toarray()
for i, col in enumerate(NUM_COLS):
    print(f"  {col:12s}: mean={arr[:, i].mean():+.3f}, std={arr[:, i].std():.3f}, min={arr[:, i].min():+.3f}, max={arr[:, i].max():+.3f}")

Median (train): years_exp = 3.0, year = 2024.0
X_num_train: (458568, 2) | X_num_test: (114773, 2)
Numeric stats (train, sau scale):
  years_exp   : mean=-0.000, std=1.000, min=-1.503, max=+3.767
  year        : mean=-0.000, std=1.000, min=-1.489, max=+1.929


### Trực quan — quan hệ numeric feature với target

Hexbin density `years_exp` và `year` (giá trị gốc, trước scale) vs `y = log1p(salary)`. Kỳ vọng `r(years_exp, y)` cao (~+0.33), `r(year, y)` thấp (~+0.07) — sẽ được xác nhận ở Section 9 sanity check.

In [ ]:
# Dùng giá trị numeric đã impute (trước scale) để trực quan đúng đơn vị thực
years_raw = num_imputer.transform(train[NUM_COLS])[:, 0]
year_raw = num_imputer.transform(train[NUM_COLS])[:, 1]

r_years = np.corrcoef(years_raw, y_train)[0, 1]
r_year = np.corrcoef(year_raw, y_train)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
hb = ax.hexbin(years_raw, y_train, gridsize=40, cmap='viridis', bins='log',
               extent=(0, 9, 0, 5))
ax.set_xlabel('years_exp (số năm KN, đã impute median=3)')
ax.set_ylabel('y = log1p(expected_salary)')
ax.set_title(f'years_exp vs y   (r = {r_years:+.4f})')
plt.colorbar(hb, ax=ax, label='log(số JD)')

ax = axes[1]
hb = ax.hexbin(year_raw, y_train, gridsize=(5, 40), cmap='viridis', bins='log',
               extent=(2021.5, 2026.5, 0, 5))
ax.set_xlabel('year (năm đăng JD)')
ax.set_ylabel('y = log1p(expected_salary)')
ax.set_title(f'year vs y   (r = {r_year:+.4f})')
ax.set_xticks([2022, 2023, 2024, 2025, 2026])
plt.colorbar(hb, ax=ax, label='log(số JD)')

plt.suptitle('Quan hệ feature numeric ↔ target  — years_exp signal mạnh, year gần như không', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fe_numeric_vs_target.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'fe_numeric_vs_target.png'}")

## 4. Categorical → One-Hot

`handle_unknown='ignore'` để giá trị lạ trong test (vd. province không có ở train) thì cột tương ứng = 0 chứ không raise.

In [5]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)
X_cat_train = ohe.fit_transform(train[CAT_COLS])
X_cat_test = ohe.transform(test[CAT_COLS])

print(f"X_cat_train: {X_cat_train.shape} | X_cat_test: {X_cat_test.shape}")
print(f"\nSố category mỗi cột (fit trên train):")
for col, cats in zip(CAT_COLS, ohe.categories_):
    print(f"  {col:18s}: {len(cats):>3}")

X_cat_train: (458568, 94) | X_cat_test: (114773, 94)

Số category mỗi cột (fit trên train):
  province          :  64
  education_level   :   7
  job_type          :   8
  job_position      :  15


## 5. Multi-label `industries_list`

Stage 2 đã chốt top-50 ngành base + `'Other'` → 51 lớp. Split `|` và đưa vào `MultiLabelBinarizer`.

Vì `industries_list` không bao giờ rỗng (stage 2 đã set fallback `'Other'`), không cần xử lý empty list.

In [6]:
def to_industry_list(s):
    return [x for x in s.split('|') if x]


train_inds = train['industries_list'].map(to_industry_list).tolist()
test_inds = test['industries_list'].map(to_industry_list).tolist()

mlb = MultiLabelBinarizer(sparse_output=True)
X_ind_train = mlb.fit_transform(train_inds).astype(np.float32)
X_ind_test = mlb.transform(test_inds).astype(np.float32)

print(f"X_ind_train: {X_ind_train.shape} | X_ind_test: {X_ind_test.shape}")
print(f"Số ngành: {len(mlb.classes_)}")
print(f"5 ngành đầu (alphabetical): {list(mlb.classes_[:5])}")

X_ind_train: (458568, 51) | X_ind_test: (114773, 51)
Số ngành: 51
5 ngành đầu (alphabetical): ['An ninh - Bảo vệ', 'An toàn lao động', 'Biên phiên dịch', 'Bán hàng - Kinh doanh', 'Bán sỉ - Bán lẻ - Quản lý cửa hàng']


## 6. TF-IDF cho 4 cột text — fit từng cột riêng với params khác nhau

Theo chốt ở [CLAUDE.md §5](../CLAUDE.md): vectorize **riêng từng cột** rồi `hstack` (không concat trước rồi vectorize 1 lần). Lý do: giữ phân biệt giữa "kinh nghiệm Python" trong `requirements` vs `benefits`.

**Params 2 loại:**
- `job_title` (text ngắn 5-20 từ): max_features=5k, **bigram (1,2)**, min_df=5 → bắt cụm "kế toán trưởng", "senior engineer"
- `job_description`, `requirements`, `benefits` (text dài): max_features=10k, unigram, min_df=10

Bước này tốn ~3-5 phút trên 459k docs × 4 cột.

In [7]:
tfidf_models = {}
text_train_blocks = []
text_test_blocks = []

for col, params in TFIDF_COLS_PARAMS:
    print(f"\n--- {col}  (ngram={params['ngram_range']}, max_features={params['max_features']:,}) ---")
    vec = TfidfVectorizer(**params)
    Xtr = vec.fit_transform(train[col]).astype(np.float32)
    Xte = vec.transform(test[col]).astype(np.float32)
    tfidf_models[col] = vec
    text_train_blocks.append(Xtr)
    text_test_blocks.append(Xte)
    nnz_per_row = Xtr.getnnz(axis=1)
    print(f"  vocab kept    : {len(vec.vocabulary_):,}")
    print(f"  X_train shape : {Xtr.shape}")
    print(f"  X_test  shape : {Xte.shape}")
    print(f"  nnz/row train : median={int(np.median(nnz_per_row))}, mean={nnz_per_row.mean():.1f}, max={nnz_per_row.max()}")


--- job_title  (ngram=(1, 2), max_features=5,000) ---


  vocab kept    : 5,000
  X_train shape : (458568, 5000)
  X_test  shape : (114773, 5000)
  nnz/row train : median=11, mean=12.2, max=60

--- job_description  (ngram=(1, 1), max_features=10,000) ---


  vocab kept    : 10,000
  X_train shape : (458568, 10000)
  X_test  shape : (114773, 10000)
  nnz/row train : median=69, mean=75.6, max=549

--- requirements  (ngram=(1, 1), max_features=10,000) ---


  vocab kept    : 9,087
  X_train shape : (458568, 9087)
  X_test  shape : (114773, 9087)
  nnz/row train : median=47, mean=49.9, max=403

--- benefits  (ngram=(1, 1), max_features=10,000) ---


  vocab kept    : 7,873
  X_train shape : (458568, 7873)
  X_test  shape : (114773, 7873)
  nnz/row train : median=55, mean=59.9, max=407


### Trực quan — top 15 token TF-IDF mỗi cột text

Token có **mean TF-IDF cao nhất** trên train — cho biết mỗi cột text nhấn vào khái niệm gì. `job_title` thường là bigram chức danh ("nhân viên", "kế toán"…), text dài thì là từ ngữ đặc trưng JD ("yêu cầu", "kinh nghiệm"…).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes_flat = axes.flat

for ax, (col, vec), block in zip(axes_flat, TFIDF_COLS_PARAMS, text_train_blocks):
    vocab = vec.get_feature_names_out()
    # mean TF-IDF qua tất cả document (sum / nrow vì matrix sparse)
    mean_tfidf = np.asarray(block.mean(axis=0)).ravel()
    top_idx = np.argsort(mean_tfidf)[-15:]
    tokens = vocab[top_idx]
    values = mean_tfidf[top_idx]

    color = 'tab:orange' if col == 'job_title' else 'steelblue'
    ax.barh(np.arange(len(tokens)), values, color=color, edgecolor='white', linewidth=0.4)
    ax.set_yticks(np.arange(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel('Mean TF-IDF')
    ax.set_title(f"{col}  (ngram={vec.ngram_range}, vocab={len(vec.vocabulary_):,})")
    for i, v in enumerate(values):
        ax.text(v * 1.01, i, f'{v:.4f}', va='center', fontsize=8)

plt.suptitle('Top 15 token TF-IDF (theo mean across train) — mỗi cột text', y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fe_tfidf_top_terms.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'fe_tfidf_top_terms.png'}")

## 7. Hstack tất cả

Thứ tự nhóm cột: `numeric → one_hot → industries → tfidf_job_title → tfidf_job_description → tfidf_requirements → tfidf_benefits`.

Lưu `groups` (start/end index của từng nhóm) vào `meta.json` để stage 4 slice ngược.

In [8]:
blocks_train = [X_num_train, X_cat_train, X_ind_train] + text_train_blocks
blocks_test = [X_num_test, X_cat_test, X_ind_test] + text_test_blocks

X_train = sparse.hstack(blocks_train, format='csr', dtype=np.float32)
X_test = sparse.hstack(blocks_test, format='csr', dtype=np.float32)

group_names = ['numeric', 'one_hot', 'industries'] + [f'tfidf_{c}' for c in TEXT_COLS]
sizes = [b.shape[1] for b in blocks_train]
ends = list(np.cumsum(sizes))
starts = [0] + ends[:-1]
groups = {name: [int(s), int(e)] for name, s, e in zip(group_names, starts, ends)}

density = X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100
print(f"X_train: {X_train.shape} | nnz = {X_train.nnz:,} | density = {density:.4f}%")
print(f"X_test : {X_test.shape}  | nnz = {X_test.nnz:,}")
print(f"\nColumn groups:")
for name, (s, e) in groups.items():
    print(f"  {name:25s}: [{s:>6}, {e:>6})  ({e-s:>5} cột)")

X_train: (458568, 32107) | nnz = 93,983,398 | density = 0.6383%
X_test : (114773, 32107)  | nnz = 23,495,995

Column groups:
  numeric                  : [     0,      2)  (    2 cột)
  one_hot                  : [     2,     96)  (   94 cột)
  industries               : [    96,    147)  (   51 cột)
  tfidf_job_title          : [   147,   5147)  ( 5000 cột)
  tfidf_job_description    : [  5147,  15147)  (10000 cột)
  tfidf_requirements       : [ 15147,  24234)  ( 9087 cột)
  tfidf_benefits           : [ 24234,  32107)  ( 7873 cột)


### Trực quan — thành phần 32k feature theo nhóm

2 góc nhìn: (1) **số cột** mỗi nhóm — TF-IDF text áp đảo (~32k/32.1k); (2) **đóng góp nnz** (non-zero entries) — vì TF-IDF sparse hơn nhiều so với numeric/OHE/multi-label (dense per row).

In [ ]:
group_sizes = {name: e - s for name, (s, e) in groups.items()}
group_nnz = {name: int(X_train[:, s:e].nnz) for name, (s, e) in groups.items()}

names = list(group_sizes.keys())
sizes_arr = np.array([group_sizes[n] for n in names])
nnz_arr = np.array([group_nnz[n] for n in names])

# Màu: numeric/categorical = warm, text TF-IDF = cool
palette = {
    'numeric': 'tab:red',
    'one_hot': 'tab:orange',
    'industries': 'gold',
    'tfidf_job_title': 'tab:cyan',
    'tfidf_job_description': 'steelblue',
    'tfidf_requirements': 'tab:blue',
    'tfidf_benefits': 'navy',
}
colors_g = [palette.get(n, 'gray') for n in names]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
y_pos = np.arange(len(names))[::-1]

ax = axes[0]
ax.barh(y_pos, sizes_arr, color=colors_g, edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('Số cột (log scale)')
ax.set_title(f'Số cột theo nhóm  (tổng = {sizes_arr.sum():,})')
for i, s in enumerate(sizes_arr):
    pct = s / sizes_arr.sum() * 100
    ax.text(s * 1.1, y_pos[i], f'{s:,}  ({pct:.2f}%)', va='center', fontsize=9)
ax.invert_yaxis()

ax = axes[1]
ax.barh(y_pos, nnz_arr, color=colors_g, edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('Số entry khác 0 trong train (log scale)')
ax.set_title(f'Đóng góp nnz  (tổng = {nnz_arr.sum():,})')
for i, n in enumerate(nnz_arr):
    pct = n / nnz_arr.sum() * 100
    ax.text(n * 1.1, y_pos[i], f'{n:,}  ({pct:.1f}%)', va='center', fontsize=9)
ax.invert_yaxis()

plt.suptitle(f'Thành phần ma trận đặc trưng — X_train {X_train.shape}, density = {density:.3f}%', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fe_group_breakdown.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Đã lưu: {FIG_DIR / 'fe_group_breakdown.png'}")

## 8. Lưu output

Vào `features/`:
- `X_{train,test}.npz` — sparse CSR.
- `y_{train,test}.npy` — vector log1p.
- `transformers.joblib` — dict các transformer đã fit.
- `meta.json` — column groups + tham số.
- `feature_names.txt` — tên đặc trưng đầy đủ (để feature importance ở stage 4).

In [9]:
feature_names = (
    list(NUM_COLS)
    + list(ohe.get_feature_names_out(CAT_COLS))
    + [f"ind__{c}" for c in mlb.classes_]
)
for col in TEXT_COLS:
    vocab = tfidf_models[col].get_feature_names_out()
    feature_names += [f"tfidf_{col}__{tok}" for tok in vocab]

assert len(feature_names) == X_train.shape[1], (len(feature_names), X_train.shape[1])

sparse.save_npz(FEATURES / 'X_train.npz', X_train)
sparse.save_npz(FEATURES / 'X_test.npz', X_test)
np.save(FEATURES / 'y_train.npy', y_train)
np.save(FEATURES / 'y_test.npy', y_test)

joblib.dump(
    {
        'num_imputer': num_imputer,
        'num_scaler': num_scaler,
        'ohe': ohe,
        'mlb': mlb,
        'tfidf': tfidf_models,
    },
    FEATURES / 'transformers.joblib',
    compress=3,
)

meta = {
    'n_train': int(X_train.shape[0]),
    'n_test': int(X_test.shape[0]),
    'n_features': int(X_train.shape[1]),
    'groups': groups,
    'num_cols': NUM_COLS,
    'cat_cols': CAT_COLS,
    'text_cols': TEXT_COLS,
    'tfidf_params': {
        'long': {**TFIDF_PARAMS_LONG, 'ngram_range': list(TFIDF_PARAMS_LONG['ngram_range'])},
        'title': {**TFIDF_PARAMS_TITLE, 'ngram_range': list(TFIDF_PARAMS_TITLE['ngram_range'])},
    },
}
(FEATURES / 'meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
(FEATURES / 'feature_names.txt').write_text('\n'.join(feature_names), encoding='utf-8')

print('Đã lưu:')
for fn in ['X_train.npz', 'X_test.npz', 'y_train.npy', 'y_test.npy',
           'transformers.joblib', 'meta.json', 'feature_names.txt']:
    p = FEATURES / fn
    print(f'  {fn:25s} {p.stat().st_size / 1e6:>7.2f} MB')

Đã lưu:
  X_train.npz                456.25 MB
  X_test.npz                 114.02 MB
  y_train.npy                  3.67 MB
  y_test.npy                   0.92 MB
  transformers.joblib          0.38 MB
  meta.json                    0.00 MB
  feature_names.txt            0.88 MB


## 9. Sanity check

Load lại từ disk, kiểm chứng shape khớp và xem correlation đơn giản giữa target và một vài feature dễ giải nghĩa (years_exp, year).

In [10]:
X_train_loaded = sparse.load_npz(FEATURES / 'X_train.npz')
y_train_loaded = np.load(FEATURES / 'y_train.npy')
meta_loaded = json.loads((FEATURES / 'meta.json').read_text(encoding='utf-8'))

assert X_train_loaded.shape == X_train.shape
assert np.allclose(y_train_loaded, y_train)
print(f"Reload OK: X_train {X_train_loaded.shape}, y_train {y_train_loaded.shape}")
print(f"Tổng số feature: {meta_loaded['n_features']:,}")

print(f"\nCorrelation y_train với feature numeric (dấu kì vọng: + cho years_exp, ? cho year):")
for i, name in enumerate(NUM_COLS):
    col = X_train_loaded[:, i].toarray().ravel()
    r = np.corrcoef(col, y_train_loaded)[0, 1]
    print(f"  {name:12s}: r = {r:+.4f}")

# Spot-check 5 ngành có lương trung bình cao nhất / thấp nhất
ind_start, ind_end = meta_loaded['groups']['industries']
ind_block = X_train_loaded[:, ind_start:ind_end].toarray().astype(bool)
ind_names = list(mlb.classes_)
means = []
for i, name in enumerate(ind_names):
    mask = ind_block[:, i]
    if mask.sum() >= 500:
        means.append((name, np.expm1(y_train_loaded[mask]).mean(), int(mask.sum())))

means.sort(key=lambda x: x[1])
print(f"\n5 ngành lương trung bình thấp nhất (n >= 500):")
for n, m, c in means[:5]:
    print(f"  {n:35s} {m:6.2f} triệu  (n={c:,})")
print(f"\n5 ngành lương trung bình cao nhất:")
for n, m, c in means[-5:]:
    print(f"  {n:35s} {m:6.2f} triệu  (n={c:,})")

Reload OK: X_train (458568, 32107), y_train (458568,)
Tổng số feature: 32,107

Correlation y_train với feature numeric (dấu kì vọng: + cho years_exp, ? cho year):
  years_exp   : r = +0.3282


  year        : r = +0.0707



5 ngành lương trung bình thấp nhất (n >= 500):
  Thực tập sinh                         6.50 triệu  (n=4,647)
  Lao động phổ thông                    9.55 triệu  (n=23,700)
  Khách sạn - Nhà hàng - Du lịch       10.75 triệu  (n=19,635)
  Bán sỉ - Bán lẻ - Quản lý cửa hàng   11.27 triệu  (n=14,760)
  Hành chính - Thư ký                  11.33 triệu  (n=18,223)

5 ngành lương trung bình cao nhất:
  IT Phần mềm                          15.78 triệu  (n=4,438)
  Luật - Pháp Lý - Tuân thủ            15.95 triệu  (n=4,291)
  Ngân hàng                            16.86 triệu  (n=8,046)
  Quản lý dự án                        18.69 triệu  (n=2,883)
  Bất động sản                         20.44 triệu  (n=8,877)


## 10. Bước tiếp theo

Sang [04_Modeling.ipynb](./04_Modeling.ipynb):
1. `sparse.load_npz` + `np.load` + `joblib.load` các artifact trong `features/`.
2. Baseline: trung bình `expected_salary` theo `job_industry × experience_level` (đọc lại từ `clean_data`).
3. Linear: Ridge / Lasso (chạy nhanh, tận dụng sparse).
4. Tree-based: RandomForest, GradientBoosting / LightGBM nếu có.
5. 5-fold CV trên stratified sample 150k (theo `job_industry × year`) → chốt hyperparam → refit full → evaluate test.
6. So sánh có/không `year` bằng slice cột theo `meta['groups']['numeric']`.